# pass@k Run Analysis

Loads samples.csv and pass_at_k.csv from a benchmark run. Supports model and temperature sweeps.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import plotly.express as px

RUN_NAME = None  # e.g. "tinker_family_pairs_t07_k1_5_10_20"
RUN_NAMES = [
    "tinker_qwen32b_t05_k1_20",
    "tinker_qwen8b_t05_k1_20",
    "tinker_llama31_8b_t05_k1_20",
    "tinker_llama33_70b_t05_k1_20",
]

candidates = [Path("results/runs"), Path("../results/runs")]
RUN_ROOT = next((p.resolve() for p in candidates if p.exists()), candidates[0].resolve())
PROJECT_ROOT = RUN_ROOT.parent.parent
EVAL_PIPELINE = PROJECT_ROOT / "eval-pipeline"
if str(EVAL_PIPELINE) not in sys.path:
    sys.path.insert(0, str(EVAL_PIPELINE))

if RUN_NAMES:
    run_dirs = [RUN_ROOT / name for name in RUN_NAMES]
elif RUN_NAME:
    run_dirs = [RUN_ROOT / RUN_NAME]
else:
    runs = sorted([p for p in RUN_ROOT.glob("*") if p.is_dir()])
    run_dirs = [runs[-1]] if runs else []

assert run_dirs, f"No run found under {RUN_ROOT}"
for run_dir in run_dirs:
    assert run_dir.exists(), f"Missing run folder: {run_dir}"

def rebuild_csvs_from_completed_model_jsons(run):
    from csv_exports import write_run_csvs
    from eval import _file_key, generate_summary

    config_path = run / "run_config.json"
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    provider = config.get("provider", "tinker")
    models = config.get("models", [])
    pass_k = config.get("pass_k", [1, 5, 10, 15, 20])

    all_results = {}
    for model in models:
        result_path = run / f"{_file_key(provider, model)}_results.json"
        if result_path.exists():
            all_results[f"{provider}/{model}"] = json.loads(result_path.read_text())

    if not all_results:
        for result_path in sorted(run.glob("*_results.json")):
            all_results[result_path.stem.removesuffix("_results")] = json.loads(result_path.read_text())

    assert all_results, f"No completed *_results.json files found in {run}"
    summary = generate_summary(all_results, execute_mode=True, metric_names=["pass_at_k", "accuracy"], pass_k=pass_k)
    (run / "summary.json").write_text(json.dumps(summary, indent=2))
    write_run_csvs(run, all_results, summary)
    return list(all_results)

for run in run_dirs:
    pass_at_k_csv = run / "pass_at_k.csv"
    samples_csv = run / "samples.csv"
    if not pass_at_k_csv.exists() or not samples_csv.exists():
        rebuilt_models = rebuild_csvs_from_completed_model_jsons(run)
        print(f"Rebuilt {run.name} from {len(rebuilt_models)} completed model result file(s).")
    assert pass_at_k_csv.exists(), f"Missing pass@k CSV: {pass_at_k_csv}"
    assert samples_csv.exists(), f"Missing samples CSV: {samples_csv}"

run_dirs


[PosixPath('/mnt/c/Users/CB/OneDrive/Documents/UCLA Math Research/computational-algebra-llm/results/runs/tinker_qwen32b_t05_k1_20'),
 PosixPath('/mnt/c/Users/CB/OneDrive/Documents/UCLA Math Research/computational-algebra-llm/results/runs/tinker_qwen8b_t05_k1_20'),
 PosixPath('/mnt/c/Users/CB/OneDrive/Documents/UCLA Math Research/computational-algebra-llm/results/runs/tinker_llama31_8b_t05_k1_20'),
 PosixPath('/mnt/c/Users/CB/OneDrive/Documents/UCLA Math Research/computational-algebra-llm/results/runs/tinker_llama33_70b_t05_k1_20')]

In [2]:
sample_frames = []
pass_at_k_frames = []
for run in run_dirs:
    sample_frame = pd.read_csv(run / "samples.csv")
    sample_frame.insert(0, "run_name", run.name)
    sample_frames.append(sample_frame)

    pass_frame = pd.read_csv(run / "pass_at_k.csv")
    pass_frame.insert(0, "run_name", run.name)
    pass_at_k_frames.append(pass_frame)

samples = pd.concat(sample_frames, ignore_index=True)
pass_at_k = pd.concat(pass_at_k_frames, ignore_index=True)
metric_cols = [c for c in pass_at_k.columns if c.startswith("pass@")]

aggregate = pass_at_k[pass_at_k["question_id"] == "__aggregate__"].copy()
aggregate


,run_name,model,question_id,temperature,num_samples,num_correct,num_attempted_samples,num_reference_failed_samples,pass@1,pass@2,...,pass@11,pass@12,pass@13,pass@14,pass@15,pass@16,pass@17,pass@18,pass@19,pass@20
67,tinker_qwen32b_t05_k1_20,tinker/Qwen/Qwen3-32B,__aggregate__,0.5,3350,1111,3350,0,0.3316,0.4090,...,0.5406,0.5467,0.5524,0.5577,0.5625,0.5671,0.5714,0.5754,0.5792,0.5828
135,tinker_qwen8b_t05_k1_20,tinker/Qwen/Qwen3-8B,__aggregate__,0.5,3350,793,3350,0,0.2367,0.2951,...,0.4228,0.4298,0.4365,0.4429,0.4489,0.4547,0.4603,0.4656,0.4707,0.4757
203,tinker_llama31_8b_t05_k1_20,tinker/meta-llama/Llama-3.1-8B-Instruct,__aggregate__,0.5,3350,502,3350,0,0.1499,0.1954,...,0.2979,0.3033,0.3085,0.3134,0.3180,0.3225,0.3268,0.3309,0.3349,0.3387
271,tinker_llama33_70b_t05_k1_20,tinker/meta-llama/Llama-3.3-70B-Instruct,__aggregate__,0.5,3350,1106,3350,0,0.3301,0.3604,...,0.4291,0.4323,0.4353,0.4380,0.4404,0.4426,0.4447,0.4465,0.4482,0.4498


In [3]:
long = aggregate.melt(
    id_vars=["model", "temperature"],
    value_vars=metric_cols,
    var_name="metric",
    value_name="pass@k",
)
long["k"] = long["metric"].str.extract(r"pass@(\d+)").astype(int)
long["temperature_label"] = "T=" + long["temperature"].astype(str)
long = long.sort_values(["model", "temperature", "k"])

fig = px.line(
    long,
    x="k",
    y="pass@k",
    color="model",
    line_dash="temperature_label",
    markers=True,
    title="Pass@k by Model and Temperature",
    labels={"k": "k", "pass@k": "pass@k", "model": "Model", "temperature_label": "Temperature"},
)
fig.update_traces(line_shape="spline")
fig.update_layout(template="plotly_white", xaxis=dict(dtick=1), yaxis=dict(range=[0, 1]))
fig.show()


In [4]:
PLOT_K = max(int(c.split("@")[1]) for c in metric_cols)
temp_df = aggregate.sort_values(["model", "temperature"]).copy()

fig = px.line(
    temp_df,
    x="temperature",
    y=f"pass@{PLOT_K}",
    color="model",
    line_dash="model",
    markers=True,
    title=f"pass@{PLOT_K} by Temperature",
    labels={"temperature": "temperature", f"pass@{PLOT_K}": "pass@k", "model": "Model"},
)
fig.update_traces(line_shape="spline")
fig.update_layout(template="plotly_white", yaxis=dict(range=[0, 1]))
fig.show()


In [5]:
by_question = pass_at_k[pass_at_k["question_id"] != "__aggregate__"].copy()
by_question.head()


,run_name,model,question_id,temperature,num_samples,num_correct,num_attempted_samples,num_reference_failed_samples,pass@1,pass@2,...,pass@11,pass@12,pass@13,pass@14,pass@15,pass@16,pass@17,pass@18,pass@19,pass@20
0,tinker_qwen32b_t05_k1_20,tinker/Qwen/Qwen3-32B,m2_001,0.5,50,18,50,0,0.36,0.5951,...,0.9965,0.9981,0.9990,0.9995,0.9997,0.9999,0.9999,1.0000,1.0000,1.0000
1,tinker_qwen32b_t05_k1_20,tinker/Qwen/Qwen3-32B,m2_002,0.5,50,35,50,0,0.70,0.9143,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,tinker_qwen32b_t05_k1_20,tinker/Qwen/Qwen3-32B,m2_003,0.5,50,44,50,0,0.88,0.9878,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
3,tinker_qwen32b_t05_k1_20,tinker/Qwen/Qwen3-32B,m2_004,0.5,50,8,50,0,0.16,0.2971,...,0.8854,0.9089,0.9281,0.9436,0.9562,0.9662,0.9741,0.9804,0.9853,0.9891
4,tinker_qwen32b_t05_k1_20,tinker/Qwen/Qwen3-32B,m2_005,0.5,50,0,50,0,0.00,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [6]:
group_cols = ["model"] + (["temperature"] if "temperature" in samples.columns else [])
agg_spec = {
    "samples": ("correct", "size"),
    "correct": ("correct", "sum"),
    "errors": ("error", lambda s: s.notna().sum()),
    "avg_total_tokens": ("total_tokens", "mean"),
}
if "oracle_raw_response" in samples.columns:
    agg_spec["oracle_calls"] = ("oracle_raw_response", lambda s: s.notna().sum())

sample_stats = samples.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()
sample_stats


,model,temperature,samples,correct,errors,avg_total_tokens,oracle_calls
0,tinker/Qwen/Qwen3-32B,0.5,3350,1111,0,1208.634627,1917
1,tinker/Qwen/Qwen3-8B,0.5,3350,793,0,1143.923284,2207
2,tinker/meta-llama/Llama-3.1-8B-Instruct,0.5,3350,502,0,293.419104,2894
3,tinker/meta-llama/Llama-3.3-70B-Instruct,0.5,3350,1106,0,286.165672,2365
